# Split mROI groups into separate outputs (no source edits)

Take a single ScanImage multi-ROI tiff and produce two stitched outputs by
subsetting `arr._rois`. ROIs in each group are stitched left-to-right in the
order listed.

Verifies:
1. each output has the expected `(width = sum of group ROI widths)`
2. one frame from each output exactly matches the corresponding x-slice of
   the full-stitched original

In [10]:
import copy
from pathlib import Path

import numpy as np
import mbo_utilities as mbo

input_path = r"D:\kbarber\2024-01-03_mk935\raw" 
output_dir = Path(r"D:\kbarber\2024-01-03_mk935\suite2p") 

# input_path = r"D:\kbarber\test" 
# output_dir = Path(r"D:\kbarber\test\rois") 

# 1-based ROI indices; each inner list becomes one stitched output file.
groups = [[1,2], [3,4]]

output_dir.mkdir(parents=True, exist_ok=True)

In [11]:
arr = mbo.imread(input_path)
print("shape:", arr.shape, "dims:", arr.dims)
print("num_rois:", arr.num_rois)
for i, r in enumerate(arr._rois, start=1):
    print(f"  roi {i}: y={r['y_start']}..{r['y_end']} (h={r['height']}), w={r['width']}")

all_idx = {i for g in groups for i in g}
assert all_idx <= set(range(1, arr.num_rois + 1)), (
    f"groups {groups} reference ROIs outside 1..{arr.num_rois}"
)

shape: (44232, 1, 29, 294, 584) dims: ('T', 'C', 'Z', 'Y', 'X')
num_rois: 4
  roi 1: y=0..294 (h=294), w=146
  roi 2: y=366..660 (h=294), w=146
  roi 3: y=732..1026 (h=294), w=146
  roi 4: y=1098..1390 (h=292), w=146


In [12]:
# Disable phase correction for a deterministic round-trip test.
# _read_pages auto-computes a per-chunk shift when no fixed shift is set,
# so a multi-frame write chunk and a single-frame verify read see different
# corrections of the same source pixels. For a real split-and-write workflow
# (not a verification test), leave fix_phase as it was.
arr.fix_phase = True  # propagates to sub copies via shared phase_correction obj

all_rois = list(arr._rois)

splits = []
for g in groups:
    sub = copy.copy(arr)
    sub._metadata = copy.deepcopy(arr._metadata)  # don't share metadata across writes
    sub._rois = [all_rois[i - 1] for i in g]
    sub.roi = None  # stitched view of just this subset
    splits.append((g, sub))

for g, sub in splits:
    expected_w = sum(all_rois[i - 1]["width"] for i in g)
    print(f"group {g}: shape={sub.shape}, expected width={expected_w}, fix_phase={sub.fix_phase}")
    assert sub.shape[-1] == expected_w, (sub.shape, expected_w)

group [1, 2]: shape=(44232, 1, 29, 294, 292), expected width=292, fix_phase=True
group [3, 4]: shape=(44232, 1, 29, 294, 292), expected width=292, fix_phase=True


In [ ]:
out_paths = []
for g, sub in splits:
    suffix = "roi" + "-".join(str(i) for i in g)
    p = mbo.imwrite(
        sub,
        output_dir,
        ext=".tif",
        output_suffix=suffix,
        overwrite=True,
    )
    out_paths.append(p)
    print(g, "->", p)

Writing TIFF:   0%|          | 0/1282728 [00:00<?, ?pg/s]

[1, 2] -> D:\kbarber\2024-01-03_mk935\suite2p\tp00001-44232_ch01_zplane01-29_roi1-2.tif


Writing TIFF:   0%|          | 0/1282728 [00:00<?, ?pg/s]

In [5]:
readbacks = []
for (g, sub), p in zip(splits, out_paths):
    rb = mbo.imread(p)
    print(f"group {g}: written shape={sub.shape}, readback shape={rb.shape}")
    assert rb.shape[-1] == sub.shape[-1], (rb.shape, sub.shape)
    readbacks.append(rb)

group [1, 2]: written shape=(743, 1, 29, 294, 292), readback shape=(743, 29, 294, 292)
group [3, 4]: written shape=(743, 1, 29, 294, 292), readback shape=(743, 29, 294, 292)


In [6]:
# Compare a single 2D Y x X plane of each output against the same x-slice of
# the full-stitched original. We read the written tiff with tifffile directly
# to skip mbo's _SqueezeSingletonDims wrapper, and squeeze both sides to 2D so
# dim-convention differences can't cause a false negative. With fix_phase=False
# on both the writes and this read, bytes should be exactly equal.

import tifffile

t, z = 0, 0
full_xslices = arr.output_xslices

arr_frame = np.asarray(arr[t]).squeeze()  # (Z, Y, X) for C=1 source
arr_2d = arr_frame[z] if arr_frame.ndim == 3 else arr_frame
print(f"arr[t={t}] full-stitch frame: {arr_frame.shape}, 2d plane (z={z}): {arr_2d.shape}")

for (g, _sub), p in zip(splits, out_paths):
    rb_raw = tifffile.imread(str(p))
    rb_t = rb_raw[t] if rb_raw.ndim >= 3 else rb_raw
    rb_2d = np.squeeze(rb_t)
    rb_2d = rb_2d[z] if rb_2d.ndim == 3 else rb_2d

    expected = np.concatenate(
        [arr_2d[..., full_xslices[i - 1]] for i in g], axis=-1
    )

    print(f"group {g}: rb raw={rb_raw.shape}, rb 2d={rb_2d.shape}, expected={expected.shape}")
    if rb_2d.shape != expected.shape:
        print(f"  SHAPE MISMATCH after squeeze. Skipping equality check.")
        continue

    ok = np.array_equal(rb_2d, expected)
    if not ok:
        diff = rb_2d.astype(np.int32) - expected.astype(np.int32)
        print(f"  max|diff|={np.abs(diff).max()}, mean|diff|={np.abs(diff).mean():.4f}")
    print(f"  equal={ok}")
    assert ok, f"group {g} mismatch"

arr[t=0] full-stitch frame: (29, 294, 584), 2d plane (z=0): (294, 584)
group [1, 2]: rb raw=(743, 29, 294, 292), rb 2d=(294, 292), expected=(294, 292)
  equal=True
group [3, 4]: rb raw=(743, 29, 294, 292), rb 2d=(294, 292), expected=(294, 292)
  equal=True
